# bangla-gemma-pro — full fine-tune of Gemma-3-1B for Bangla

**Kaggle, GPU T4 ×2.** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.
Add your Hugging Face write token as a Kaggle Secret named `HF_TOKEN`
(Add-ons → Secrets). Then Run All. Roughly 4–6 hours.

### What this does

| Stage | |
|---|---|
| Baseline | `google/gemma-3-1b-it` on five Bangla benchmarks, before any training |
| Train | **full fine-tune** (every parameter), not LoRA, via Unsloth |
| Evaluate | the same five benchmarks, plus perplexity and tokenizer efficiency |
| Publish | merged weights, GGUF for llama.cpp, and a model card built from the numbers |

### How the evaluation works, and why it is trustworthy

Bangla generation is hard to score automatically — ROUGE and BLEU behave badly on a
morphologically rich language. So the headline numbers come from **multiple-choice
benchmarks scored by log-likelihood**: for each option, the model's average log
probability is computed and the highest wins. No generation, no judge, no string
matching. The answer is either right or wrong.

The five benchmarks are the suite TituLLM uses, all from the `hishab` organisation:
`bangla-mmlu`, `commonsenseqa-bn`, `openbookqa-bn`, `piqa-bn`, `boolq_bn`.

### On comparing with other Bangla models

TigerLLM (1B) reports 0.61 on MMLU-bn and 0.59 on BEnQA. Those numbers are **not
directly comparable** to anything measured here: they use the Bangla subset of MMLU-Pro,
not `hishab/bangla-mmlu`, and the ACL version of that paper notes the results are not
reproducible. The card keeps measured numbers and cited numbers in separate tables.

## 1. Environment

In [ ]:
import subprocess, os
print(subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total",
                               "--format=csv,noheader"]).decode().strip())
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # Unsloth is single-GPU; T4 #2 stays idle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import shutil
print("disk free in /kaggle/working:",
      round(shutil.disk_usage("/kaggle/working").free/1e9, 1), "GB")

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps --upgrade git+https://github.com/unslothai/unsloth.git
!pip install -q sacrebleu matplotlib
print("installed")

## 2. Configuration

**Full fine-tuning on a 16 GB T4.** Gemma-3-1B is about 2 GB in fp16; gradients add
another 2 GB. Plain AdamW would need ~8 GB more for its two fp32 moment buffers, which
does not fit. `adamw_8bit` stores them in 8 bits instead, bringing the optimizer down to
roughly 2 GB. With gradient checkpointing the run sits near 11 GB.

This is the one place where a 1B model makes full fine-tuning honest on free hardware —
at 3B it would not fit without quantising the base, at which point it is QLoRA.

In [ ]:
import gc, re, json, time, random, inspect, math
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from unsloth import FastModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer

BASE_MODEL  = "unsloth/gemma-3-1b-it"          # mirror of google/gemma-3-1b-it, ungated
UPSTREAM    = "google/gemma-3-1b-it"
SFT_DATASET = "md-nishat-008/Bangla-Instruct"  # 100K native Bangla instructions
# perplexity uses held-out Bangla-Instruct rows, not xlsum: xlsum is script-based
# and recent `datasets` refuses to execute loading scripts.
PPL_FROM, PPL_TO = 60000, 60500

MAX_SEQ   = 512        # median example is 314 tokens; 512 keeps 80% of them
BATCH     = 2
ACCUM     = 8                                  # effective 16
MAX_STEPS = 500        # 1200 took 8 h and risks Kaggle's 9 h cut-off
LR        = 1e-5                               # full fine-tuning: an order below LoRA's 2e-4
SEED      = 3407

BENCH_N   = 300                                # per benchmark
PPL_N     = 200
MODEL_NAME = "bangla-gemma-pro"

WORK   = "/kaggle/working"
SAVE   = f"{WORK}/{MODEL_NAME}"
MERGED = f"{WORK}/{MODEL_NAME}-merged"
GGUF   = f"{WORK}/{MODEL_NAME}-GGUF"
ASSETS = f"{SAVE}/assets"
for d in (SAVE, ASSETS):
    os.makedirs(d, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
GPU, BF16 = torch.cuda.get_device_name(0), is_bfloat16_supported()
print(f"{GPU} | bfloat16 {BF16} -> training in {'bf16' if BF16 else 'fp16'}")

plt.rcParams.update({"figure.dpi":130,"savefig.dpi":130,"savefig.bbox":"tight","font.size":10,
                     "axes.spines.top":False,"axes.spines.right":False,"axes.grid":True,
                     "grid.alpha":.25,"figure.facecolor":"white"})
C_BASE, C_TUNED = "#94a3b8", "#059669"

def vram(): return torch.cuda.max_memory_allocated()/1e9
def clear(): gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN
from huggingface_hub import login, create_repo, upload_folder, whoami
login(token=HF_TOKEN)
HF_USERNAME = whoami()["name"]
REPO_ID   = f"{HF_USERNAME}/{MODEL_NAME}"
GGUF_REPO = f"{HF_USERNAME}/{MODEL_NAME}-GGUF"
print(f"logged in as {HF_USERNAME} -> {REPO_ID}")

## 3. Load the model

In [ ]:
clear()
model, tokenizer = FastModel.from_pretrained(
    model_name      = BASE_MODEL,
    max_seq_length  = MAX_SEQ,
    dtype           = None,
    load_in_4bit    = False,
    full_finetuning = True,        # every parameter trains
)
N_PARAMS = sum(p.numel() for p in model.parameters())
TRAINABLE = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"{N_PARAMS/1e9:.2f}B params | trainable {TRAINABLE/1e9:.2f}B "
      f"({100*TRAINABLE/N_PARAMS:.1f}%) | {vram():.2f} GB")
assert TRAINABLE > 0.9 * N_PARAMS, \
    "Not a full fine-tune — full_finetuning=True did not take effect."

tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

SYSTEM_BN = "আপনি একজন সহায়ক সহকারী। সবসময় স্পষ্ট ও সঠিক বাংলায় উত্তর দিন।"

def to_chat(user, assistant=None, system=SYSTEM_BN):
    msgs = []
    if system:
        msgs.append({"role": "system", "content": system})
    msgs.append({"role": "user", "content": user})
    if assistant is not None:
        msgs.append({"role": "assistant", "content": assistant})
    return msgs

## 4. Tokenizer efficiency on Bangla

A detail most Bangla model cards skip. Gemma-3 ships a 262k-token vocabulary covering
140+ languages, so Bangla should fragment less than it does under an English-centric
tokenizer. Worth measuring rather than assuming, because tokens per word sets both the
effective context length and the cost of every request.

In [ ]:
BN_SAMPLES = [
    "বাংলাদেশের রাজধানী ঢাকা একটি জনবহুল শহর।",
    "বিজ্ঞান ও প্রযুক্তির অগ্রগতি আমাদের জীবনযাত্রাকে সহজ করেছে।",
    "তিনি প্রতিদিন সকালে নদীর ধারে হাঁটতে যেতেন এবং পাখির ডাক শুনতেন।",
    "অর্থনৈতিক উন্নয়নের জন্য শিক্ষার বিকল্প নেই।",
]
tok_counts, word_counts = [], []
for s in BN_SAMPLES:
    tok_counts.append(len(tokenizer(s, add_special_tokens=False)["input_ids"]))
    word_counts.append(len(s.split()))
TOKENS_PER_WORD = sum(tok_counts)/sum(word_counts)
print(f"Bangla: {sum(tok_counts)} tokens for {sum(word_counts)} words "
      f"-> {TOKENS_PER_WORD:.2f} tokens/word")
en = "The capital of Bangladesh, Dhaka, is a densely populated city."
en_tpw = len(tokenizer(en, add_special_tokens=False)["input_ids"])/len(en.split())
print(f"English comparison: {en_tpw:.2f} tokens/word")
print(f"Bangla costs {TOKENS_PER_WORD/en_tpw:.1f}x more tokens per word than English")

## 5. Benchmark harness

Each dataset is shaped differently, so columns are detected rather than assumed, and the
detected mapping is printed so a wrong guess is visible immediately.

Scoring is length-normalised log-likelihood: the average log probability of each option's
tokens given the question. Length normalisation matters because without it the model
simply prefers whichever option is shortest.

In [ ]:
BENCHMARKS = [
    ("bangla-mmlu",      "hishab/bangla-mmlu"),
    ("commonsenseqa-bn", "hishab/commonsenseqa-bn"),
    ("openbookqa-bn",    "hishab/openbookqa-bn"),
    ("piqa-bn",          "hishab/piqa-bn"),
    ("boolq-bn",         "hishab/boolq_bn"),
]

def rows_of(repo):
    for split in ("test", "validation", "train"):
        try:
            return load_dataset(repo, split=split).shuffle(seed=SEED), split
        except Exception:
            continue
    return None, None

def adapt_mmlu(r):
    """MMLU / CommonsenseQA / OpenBookQA shapes: a choice list plus a key or index."""
    opts = r.get("choices") or r.get("options")
    if isinstance(opts, dict):
        opts = list(opts.get("text", []))
    if not opts or len(opts) < 2:
        return None
    a = r.get("answer", r.get("answerKey", r.get("label")))
    if isinstance(a, str) and len(a) == 1 and a.upper() in "ABCDE":
        gi = "ABCDE".index(a.upper())
    elif str(a).isdigit():
        gi = int(a)
    else:
        gi = next((i for i, o in enumerate(opts) if str(o).strip() == str(a).strip()), None)
    q = r.get("question") or r.get("query")
    if gi is None or not q or gi >= len(opts):
        return None
    return {"q": str(q), "ctx": "", "options": [str(o) for o in opts], "gold": gi}

def adapt_piqa(r):
    if not (r.get("sol1") and r.get("sol2")):
        return None
    return {"q": str(r.get("goal") or r.get("question") or ""), "ctx": "",
            "options": [str(r["sol1"]), str(r["sol2"])], "gold": int(r.get("label", 0))}

def adapt_boolq(r):
    """boolq_bn stores a boolean or a yes/no string, which no generic detector maps."""
    a = r.get("answer")
    if isinstance(a, bool):
        gi = 0 if a else 1
    else:
        t = str(a).strip().lower()
        gi = 0 if t in ("true", "yes", "হ্যাঁ", "হ্যা") else (
             1 if t in ("false", "no", "না") else None)
    q = r.get("question")
    if gi is None or not q:
        return None
    return {"q": str(q), "ctx": str(r.get("passage") or ""),
            "options": ["হ্যাঁ", "না"], "gold": gi}

# name, repo, adapter, random-chance floor
BENCHMARKS = [
    ("bangla-mmlu",      "hishab/bangla-mmlu",      adapt_mmlu, 0.25),
    ("commonsenseqa-bn", "hishab/commonsenseqa-bn", adapt_mmlu, 0.20),
    ("openbookqa-bn",    "hishab/openbookqa-bn",    adapt_mmlu, 0.25),
    ("piqa-bn",          "hishab/piqa-bn",          adapt_piqa, 0.50),
    ("boolq-bn",         "hishab/boolq_bn",         adapt_boolq, 0.50),
]

print("loading benchmarks")
BENCH, CHANCE = {}, {}
for name, repo, adapt, chance in BENCHMARKS:
    ds, split = rows_of(repo)
    if ds is None:
        print(f"  {name}: could not load"); continue
    items = []
    for r in ds:
        it = adapt(r)
        if it:
            items.append(it)
        if len(items) >= BENCH_N:
            break
    if not items:
        print(f"  {name}: adapter produced nothing; columns were {ds.column_names}")
        print(f"     first row: { {k: str(v)[:45] for k, v in ds[0].items()} }")
        continue
    BENCH[name], CHANCE[name] = items, chance
    print(f"  {name}: {len(items)} items, {len(items[0]['options'])} options, "
          f"chance {chance:.2f}, split={split}")

assert BENCH, "No benchmark loaded — is Internet enabled in the notebook settings?"
print(f"\n{len(BENCH)} benchmarks ready: {list(BENCH)}")

In [ ]:
@torch.no_grad()
def option_logprob(prompt_ids, option_text):
    opt_ids = tokenizer(option_text, add_special_tokens=False)["input_ids"]
    if not opt_ids:
        return -1e9
    ids = torch.tensor([prompt_ids + opt_ids], device="cuda")
    labels = ids.clone()
    labels[:, :len(prompt_ids)] = -100          # score the option only
    out = model(input_ids=ids, labels=labels)
    return -float(out.loss)                     # mean log prob per option token

@torch.no_grad()
def run_benchmarks(tag):
    model.eval()
    results = {}
    for name, items in BENCH.items():
        t0, correct = time.time(), 0
        for it in items:
            user = (f"{it['ctx']}\n\n{it['q']}" if it["ctx"] else it["q"])
            prompt = tokenizer.apply_chat_template(to_chat(user), tokenize=False,
                                                   add_generation_prompt=True)
            pids = tokenizer(prompt, add_special_tokens=False)["input_ids"][-(MAX_SEQ-64):]
            scores = [option_logprob(pids, o) for o in it["options"]]
            correct += int(int(np.argmax(scores)) == it["gold"])
        acc = correct/len(items)
        results[name] = round(acc, 4)
        print(f"  {tag} {name:18} {acc:.3f}  ({time.time()-t0:.0f}s)")
    results["average"] = round(float(np.mean([v for k, v in results.items()
                                              if k != "average"])), 4)
    print(f"  {tag} {'AVERAGE':18} {results['average']:.3f}")
    return results

@torch.no_grad()
def perplexity(texts):
    model.eval()
    nll, ntok = 0.0, 0
    for t in texts:
        ids = tokenizer(t, add_special_tokens=False, truncation=True,
                        max_length=MAX_SEQ)["input_ids"]
        if len(ids) < 16:
            continue
        x = torch.tensor([ids], device="cuda")
        loss = model(input_ids=x, labels=x).loss
        nll += float(loss)*(len(ids)-1); ntok += len(ids)-1
    return round(math.exp(nll/max(ntok,1)), 2)

# Held-out Bangla for perplexity. Training uses the first 40,000 rows of the shuffled
# dataset, so rows 60,000+ have certainly not been seen.
_ppl_raw = load_dataset(SFT_DATASET, split="train").shuffle(seed=SEED)
PPL_TEXTS = [str(r["response"]).strip()
             for r in _ppl_raw.select(range(PPL_FROM, PPL_TO))
             if len(str(r["response"]).split()) >= 25][:PPL_N]
del _ppl_raw
print(f"\nperplexity corpus: {len(PPL_TEXTS)} held-out Bangla responses, median "
      f"{int(np.median([len(t.split()) for t in PPL_TEXTS]))} words")
assert len(PPL_TEXTS) >= 50, "not enough held-out text for a stable perplexity"

## 6. Baseline

The number everything else is judged against. Gemma-3-1B-it is already multilingual, so
the question is not whether it can read Bangla but whether this fine-tune improves it.

In [ ]:
clear()
print("baseline:")
BASE_BENCH = run_benchmarks("base")
BASE_PPL = perplexity(PPL_TEXTS) if PPL_TEXTS else None
print("baseline perplexity:", BASE_PPL)

PROBES = [
    "বাংলাদেশের মুক্তিযুদ্ধ সম্পর্কে তিনটি বাক্যে লিখুন।",
    "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?",
    "একটি ছোট গল্প লিখুন যেখানে একজন কৃষক বৃষ্টির জন্য অপেক্ষা করছেন।",
    "পানি ফুটলে কী ঘটে, সহজ ভাষায় ব্যাখ্যা করুন।",
]

@torch.no_grad()
def generate(texts, max_new=200):
    model.eval()
    outs = []
    for t in texts:
        p = tokenizer.apply_chat_template(to_chat(t), tokenize=False,
                                          add_generation_prompt=True)
        enc = tokenizer(p, return_tensors="pt", add_special_tokens=False).to("cuda")
        g = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
        outs.append(tokenizer.decode(g[0][enc["input_ids"].shape[1]:],
                                     skip_special_tokens=True).strip())
    return outs

BASE_GEN = generate(PROBES)
for q, a in zip(PROBES, BASE_GEN):
    print(f"\nপ্রশ্ন: {q}\nবেস: {a[:260]}")

# The baseline takes ~50 minutes. Save it so a later crash does not cost it again.
json.dump({"base": BASE_BENCH, "base_ppl": BASE_PPL, "chance": CHANCE,
           "base_gen": dict(zip(PROBES, BASE_GEN)),
           "tokens_per_word": {"bangla": TOKENS_PER_WORD, "english": en_tpw}},
          open(f"{SAVE}/baseline.json", "w"), indent=2, ensure_ascii=False)
print(f"\nbaseline saved to {SAVE}/baseline.json")
clear()

## 7. Training data

`Bangla-Instruct` is 100K instruction-response pairs generated **natively in Bangla**
rather than machine-translated from English. That distinction matters: the translated
Alpaca derivatives most Bangla fine-tunes use inherit English sentence structure and
translation artefacts, and the TigerLLM paper attributes much of its advantage to using
native data instead.

In [ ]:
raw = load_dataset(SFT_DATASET, split="train")
print(raw)
cols = raw.column_names
INS_COL = next(c for c in ["instruction","prompt","question","input_text"] if c in cols)
OUT_COL = next(c for c in ["response","output","answer","completion"] if c in cols)
CTX_COL = next((c for c in ["input","context"] if c in cols), None)
print(f"instruction: {INS_COL} | response: {OUT_COL} | extra input: {CTX_COL}")

raw = raw.shuffle(seed=SEED)

def build(ex):
    user = str(ex[INS_COL]).strip()
    if CTX_COL and str(ex.get(CTX_COL) or "").strip():
        user += "\n\n" + str(ex[CTX_COL]).strip()
    return {"text": tokenizer.apply_chat_template(
        to_chat(user, str(ex[OUT_COL]).strip()), tokenize=False)}

ds = raw.select(range(min(40000, len(raw)))).map(
    build, remove_columns=raw.column_names, num_proc=2)

lens = [len(tokenizer(t, add_special_tokens=False)["input_ids"]) for t in ds["text"][:2000]]
print(f"\ntoken length: median {int(np.median(lens))}, 90th {int(np.percentile(lens,90))}")
before = len(ds)
ds = ds.filter(lambda e: len(tokenizer(e["text"], add_special_tokens=False)["input_ids"])
               <= MAX_SEQ, num_proc=2)
print(f"kept {len(ds):,} of {before:,} within {MAX_SEQ} tokens")
print("\n--- formatted example ---\n", ds[0]["text"][:800])

## 8. Trainer

Loss on the assistant turn only. `max_grad_norm=1.0` and a low learning rate matter more
in full fine-tuning than in LoRA: every weight moves, so a bad step damages the
multilingual ability the base model arrived with.

In [ ]:
try:
    from trl import SFTConfig
    CFG = SFTConfig
except ImportError:
    from transformers import TrainingArguments as CFG
_P = set(inspect.signature(CFG.__init__).parameters)

wanted = dict(
    per_device_train_batch_size=BATCH, gradient_accumulation_steps=ACCUM,
    warmup_steps=40, max_steps=MAX_STEPS, learning_rate=LR,
    fp16=not BF16, bf16=BF16, logging_steps=20, optim="adamw_8bit",
    weight_decay=0.01, lr_scheduler_type="cosine", max_grad_norm=1.0,
    seed=SEED, output_dir=f"{WORK}/outputs", report_to="none",
    save_strategy="no", gradient_checkpointing=True,
    max_seq_length=MAX_SEQ, dataset_text_field="text",
    dataset_num_proc=2, packing=False,
)
kw = {k: v for k, v in wanted.items() if k in _P}
extra = {k: wanted[k] for k in ["max_seq_length","dataset_text_field",
                                "dataset_num_proc","packing"] if k not in _P}
dropped = sorted(set(wanted) - set(kw) - set(extra))
if dropped:
    print("not supported by this version, skipped:", dropped)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
                     args=CFG(**kw), **extra)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n")

lab = np.array(trainer.train_dataset[0]["labels"])
sup = int((lab != -100).sum())
print(f"\n{sup}/{len(lab)} tokens supervised ({sup/len(lab):.1%})")
print("supervised begins:", repr(tokenizer.decode([t for t in lab if t != -100][:40])))
assert 0 < sup < len(lab), "response-only masking failed — check the turn markers."

## 9. Train

Around 3 hours for 1200 steps. Kaggle GPU sessions run up to 9 hours, so there is room,
but do not close the browser tab.

In [ ]:
# A previous crashed run can leave checkpoints behind; Kaggle caps /kaggle/working at 20 GB
# and a full-finetune checkpoint is ~4 GB with optimizer state.
shutil.rmtree(f"{WORK}/outputs", ignore_errors=True)
free = shutil.disk_usage(WORK).free / 1e9
print(f"disk free before training: {free:.1f} GB")
assert free > 8, "Need at least 8 GB free; clear old outputs first."

clear()
t0 = time.time()
stats = trainer.train()
TRAIN_MIN = (time.time()-t0)/60
VRAM_TRAIN = vram()
CURVE = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"\n{TRAIN_MIN:.0f} min | peak VRAM {VRAM_TRAIN:.2f} GB | final loss {stats.training_loss:.4f}")
assert not np.isnan(stats.training_loss), "loss is NaN — lower the learning rate"

## 10. Evaluate the fine-tune

In [ ]:
clear()
print("fine-tuned:")
TUNED_BENCH = run_benchmarks("tuned")
TUNED_PPL = perplexity(PPL_TEXTS) if PPL_TEXTS else None
TUNED_GEN = generate(PROBES)

D_AVG = TUNED_BENCH["average"] - BASE_BENCH["average"]
print(f"\n{'benchmark':20}{'base':>8}{'tuned':>8}{'delta':>8}")
for k in BASE_BENCH:
    print(f"{k:20}{BASE_BENCH[k]:8.3f}{TUNED_BENCH[k]:8.3f}{TUNED_BENCH[k]-BASE_BENCH[k]:+8.3f}")
if BASE_PPL:
    print(f"{'perplexity':20}{BASE_PPL:8.2f}{TUNED_PPL:8.2f}{TUNED_PPL-BASE_PPL:+8.2f}  (lower is better)")

for q, b, t in zip(PROBES, BASE_GEN, TUNED_GEN):
    print(f"\nপ্রশ্ন : {q}\nবেস    : {b[:220]}\nটিউনড  : {t[:220]}")

EXAMPLES = [{"q": q, "base": b, "tuned": t}
            for q, b, t in zip(PROBES, BASE_GEN, TUNED_GEN)]

## 11. Charts

In [ ]:
names = [k for k in BASE_BENCH if k != "average"]
fig, ax = plt.subplots(figsize=(7.6, 3.6))
x = np.arange(len(names)); w = 0.36
bv = [BASE_BENCH[k] for k in names]; tv = [TUNED_BENCH[k] for k in names]
ax.bar(x-w/2, bv, w, label="gemma-3-1b-it", color=C_BASE)
ax.bar(x+w/2, tv, w, label="bangla-gemma-pro", color=C_TUNED)
for xi, v in zip(x-w/2, bv): ax.text(xi, v+.012, f"{v:.2f}", ha="center", fontsize=8)
for xi, v in zip(x+w/2, tv): ax.text(xi, v+.012, f"{v:.2f}", ha="center", fontsize=8)
for i, n in enumerate(names):
    c = CHANCE.get(n)
    if c:
        ax.hlines(c, i-.46, i+.46, color="#dc2626", ls="--", lw=1.6,
                  label="random chance" if i == 0 else None)
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha="right")
ax.set_ylabel("accuracy"); ax.set_ylim(0, max(bv+tv+[0.6])*1.25)
ax.set_title(f"Bangla benchmarks, {BENCH_N} items each, log-likelihood scoring",
             fontsize=11, pad=10)
ax.legend(frameon=False, fontsize=9, ncol=3)
plt.savefig(f"{ASSETS}/benchmarks.png"); plt.close()

fig, ax = plt.subplots(figsize=(4.6, 3.3))
bars = ax.bar(["base","fine-tuned"], [BASE_BENCH["average"], TUNED_BENCH["average"]],
              color=[C_BASE, C_TUNED], width=.5)
for b, v in zip(bars, [BASE_BENCH["average"], TUNED_BENCH["average"]]):
    ax.text(b.get_x()+b.get_width()/2, v+.008, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("mean accuracy across five benchmarks")
ax.set_ylim(0, max(BASE_BENCH["average"], TUNED_BENCH["average"])*1.25)
ax.set_title("Average", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/average.png"); plt.close()

fig, ax = plt.subplots(figsize=(6.2, 3.0))
ax.plot([s for s,_ in CURVE], [l for _,l in CURVE], color=C_TUNED, lw=1.5)
ax.set_xlabel("step"); ax.set_ylabel("training loss")
ax.set_title("Training loss (assistant turn only)", fontsize=11, pad=10)
plt.savefig(f"{ASSETS}/loss.png"); plt.close()

if BASE_PPL:
    fig, ax = plt.subplots(figsize=(5.6, 3.2))
    bars = ax.bar(["base","fine-tuned"], [BASE_PPL, TUNED_PPL],
                  color=[C_BASE, C_TUNED], width=.5)
    for b, v in zip(bars, [BASE_PPL, TUNED_PPL]):
        ax.text(b.get_x()+b.get_width()/2, v*1.01, f"{v:.1f}", ha="center", fontsize=10)
    ax.set_ylabel("perplexity (lower is better)")
    ax.set_title("Perplexity on held-out Bangla prose", fontsize=11, pad=10)
    plt.savefig(f"{ASSETS}/perplexity.png"); plt.close()

print("charts:", sorted(os.listdir(ASSETS)))
from IPython.display import Image, display
display(Image(f"{ASSETS}/benchmarks.png"))

## 12. Save weights and GGUF

Kaggle's working directory is capped at 20 GB. A 1B model is about 2 GB merged and under
1 GB in GGUF, so both fit comfortably.

In [ ]:
model.save_pretrained(SAVE); tokenizer.save_pretrained(SAVE)
try:
    model.save_pretrained_merged(MERGED, tokenizer, save_method="merged_16bit")
except Exception:
    shutil.copytree(SAVE, MERGED, dirs_exist_ok=True)   # full fine-tune: already merged
merged_gb = sum(os.path.getsize(f"{MERGED}/{f}") for f in os.listdir(MERGED)
                if os.path.isfile(f"{MERGED}/{f}"))/1e9
print(f"merged: {merged_gb:.2f} GB")

GGUF_OK, GGUF_FILES, GGUF_ERR = False, [], None
try:
    t0 = time.time()
    model.save_pretrained_gguf(GGUF, tokenizer, quantization_method=["q4_k_m", "q8_0"])
    print(f"converted in {(time.time()-t0)/60:.0f} min")
except Exception as e:
    GGUF_ERR = str(e)[:300]
    print("conversion raised:", GGUF_ERR)

# Unsloth writes to "<dir>_gguf" and names the files after the BASE model, so collect from
# every plausible location and rename them after this model.
import glob
found = sorted({p for d in [GGUF, GGUF + "_gguf", WORK]
                if os.path.isdir(d) for p in glob.glob(os.path.join(d, "*.gguf"))})
os.makedirs(GGUF, exist_ok=True)
for src_f in found:
    m = re.search(r"\.(Q\d[^.]*|F16|BF16)\.gguf$", os.path.basename(src_f), re.I)
    quant = m.group(1) if m else os.path.basename(src_f).split(".")[-2]
    dst = os.path.join(GGUF, f"{MODEL_NAME}.{quant}.gguf")
    if os.path.abspath(src_f) != os.path.abspath(dst):
        shutil.move(src_f, dst)
    GGUF_FILES.append(os.path.basename(dst))

for f in list(GGUF_FILES):          # F16 is ~2 GB and redundant next to Q8_0
    if re.search(r"\.(F16|BF16)\.gguf$", f, re.I):
        os.remove(os.path.join(GGUF, f)); GGUF_FILES.remove(f)

GGUF_FILES = sorted(set(GGUF_FILES))
GGUF_OK = bool(GGUF_FILES)
for f in GGUF_FILES:
    print(f"  {f}: {os.path.getsize(os.path.join(GGUF, f))/1e9:.2f} GB")
print("GGUF_OK =", GGUF_OK)
if not GGUF_OK:
    print("The merged weights are saved; convert later with llama.cpp if needed.")

json.dump({"chat_template":"gemma-3","system":SYSTEM_BN},
          open(f"{SAVE}/prompt_format.json","w"), indent=2, ensure_ascii=False)

## 13. Model card

In [ ]:
def cl(s, n=90):
    s = " ".join(str(s).split())[:n].rstrip("\\")
    return s.replace("|","\\|")

bench_rows = "\n".join(
    f"| {k} | {CHANCE.get(k, 0):.2f} | {BASE_BENCH[k]:.3f} | **{TUNED_BENCH[k]:.3f}** | "
    f"{TUNED_BENCH[k]-BASE_BENCH[k]:+.3f} |" for k in names)
PPL_PCT = 100 * (BASE_PPL - TUNED_PPL) / BASE_PPL if BASE_PPL else 0.0
ex_rows = "\n".join(f"| {cl(e['q'],60)} | {cl(e['base'],90)} | {cl(e['tuned'],90)} |"
                    for e in EXAMPLES)
gguf_rows = ("\n".join(f"| `{f}` | {os.path.getsize(f'{GGUF}/{f}')/1e9:.2f} GB |"
                       for f in GGUF_FILES) if GGUF_OK
             else "| — | GGUF conversion did not complete in this run |")
ppl_row = (f"| perplexity on held-out Bangla | {BASE_PPL:.2f} | **{TUNED_PPL:.2f}** | "
           f"{TUNED_PPL-BASE_PPL:+.2f} (lower is better) |" if BASE_PPL else "")

if PPL_PCT > 10:
    headline = (f"**Perplexity on held-out Bangla falls from {BASE_PPL:.2f} to "
                f"{TUNED_PPL:.2f} — a {PPL_PCT:.0f}% reduction.** That is the headline "
                "because it is the only measure here without a chance floor: the "
                "multiple-choice benchmarks below sit near theirs for both models and "
                "separate almost nothing, while perplexity directly measures how well the "
                "model predicts Bangla it has never seen.")
elif PPL_PCT > 2:
    headline = (f"**Perplexity on held-out Bangla improves from {BASE_PPL:.2f} to "
                f"{TUNED_PPL:.2f}**, a {PPL_PCT:.0f}% reduction — a real if modest gain, and "
                "the only measure here with no chance floor.")
elif PPL_PCT > -2:
    headline = (f"Perplexity is essentially unchanged ({BASE_PPL:.2f} → {TUNED_PPL:.2f}) and "
                "the multiple-choice benchmarks sit near their chance floor for both models. "
                "What this run establishes is the pipeline and the tokenizer finding below "
                "rather than a better model.")
else:
    headline = (f"Perplexity rose from {BASE_PPL:.2f} to {TUNED_PPL:.2f}: the model predicts "
                "held-out Bangla worse than its base. Published as a measured result.")

CARD = """---
language:
  - bn
  - en
license: gemma
library_name: transformers
pipeline_tag: text-generation
base_model: __UPSTREAM__
base_model_relation: finetune
tags:
  - bangla
  - bengali
  - gemma3
  - unsloth
  - full-finetune
  - instruction-tuned
datasets:
  - md-nishat-008/Bangla-Instruct
metrics:
  - accuracy
  - perplexity
model-index:
  - name: __MODEL_NAME__
    results:
      - task:
          type: text-generation
          name: Bangla multiple-choice understanding
        dataset:
          name: Bangla benchmark suite (bangla-mmlu, commonsenseqa-bn, openbookqa-bn, piqa-bn, boolq-bn)
          type: hishab/bangla-mmlu
        metrics:
          - type: accuracy
            name: Mean accuracy across five benchmarks
            value: __AVG__
---

<img src="assets/banner.png" alt="__MODEL_NAME__" width="100%">

# __MODEL_NAME__

**A full fine-tune of [`__UPSTREAM__`](https://huggingface.co/__UPSTREAM__) for Bangla.**
Every parameter was trained — not LoRA, not an adapter — on
[`Bangla-Instruct`](https://huggingface.co/datasets/md-nishat-008/Bangla-Instruct),
100K instruction-response pairs written natively in Bangla rather than translated from
English.

__HEADLINE__

Trained on **free Kaggle T4** hardware in __MINS__ minutes, peaking at __VRAM__ GB of the
T4's 15 GB.

## Results

Five Bangla multiple-choice benchmarks, __BENCH_N__ items each, scored by
**length-normalised log-likelihood** — the model's average log probability for each option,
highest wins. No generation, no LLM judge, no string matching: each answer is simply right
or wrong.

![Benchmarks](assets/benchmarks.png)

Perplexity on __PPLN__ held-out `Bangla-Instruct` responses, drawn from well past the rows
used in training:

![Perplexity](assets/perplexity.png)

| | perplexity |
|---|---|
| `gemma-3-1b-it` | __BASE_PPL__ |
| **__MODEL_NAME__** | **__TUNED_PPL__** |

Multiple choice, scored by length-normalised log-likelihood:

| Benchmark | Random chance | `gemma-3-1b-it` | **this model** | Δ |
|---|---|---|---|---|
__BENCH_ROWS__
| **average** | — | __BASE_AVG__ | **__AVG__** | __D_AVG__ |

**The chance column changes how that table reads.** At 1B scale these benchmarks sit close
to guessing for both models, so the differences between columns carry little information.
They are reported because they were run, and because quoting an accuracy without its floor
is quoting a number nobody can interpret.

![Average](assets/average.png)



### Comparison with other Bangla models

Numbers published by other Bangla LLM projects are listed here **for context only, not as a
like-for-like comparison**:

| Model | Reported | Benchmark used |
|---|---|---|
| TigerLLM (1B) | 0.61 | MMLU-Pro Bangla subset |
| TigerLLM (1B) | 0.59 | BEnQA |

Those figures come from a different MMLU variant than the `hishab/bangla-mmlu` used here,
and the ACL version of that paper notes the reported results are not reproducible. Putting
them in the same column as the measurements above would be dishonest, so they are kept
separate. If you want a direct comparison, run the harness in this notebook against both
models on the same five datasets.

### Sample generations

| Prompt | `gemma-3-1b-it` | This model |
|---|---|---|
__EX_ROWS__

![Training loss](assets/loss.png)

## Tokenizer efficiency on Bangla

Gemma-3 ships a 262k-token vocabulary covering 140+ languages, which matters for a Bangla
model: tokens per word sets both the usable context length and the cost of every request.

| | tokens per word |
|---|---|
| Bangla | __TPW__ |
| English | __ETPW__ |

Bangla costs **__RATIO__×** as many tokens per word as English here. Budget context length
accordingly: a 1,024-token window holds noticeably less Bangla than the same window of
English.

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tok = AutoTokenizer.from_pretrained("__REPO_ID__")
model = AutoModelForCausalLM.from_pretrained("__REPO_ID__", torch_dtype=torch.float16,
                                             device_map="auto")

messages = [
    {"role": "system", "content": "__SYSTEM__"},
    {"role": "user", "content": "বাংলাদেশের মুক্তিযুদ্ধ সম্পর্কে তিনটি বাক্যে লিখুন।"},
]
prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
```

The Bangla system prompt was present in every training example and is in
[`prompt_format.json`](prompt_format.json). Results degrade without it.

### llama.cpp

| File | Size |
|---|---|
__GGUF_ROWS__

```bash
llama-cli -hf __GGUF_REPO__:Q4_K_M -p "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?" -n 256
```

## Training

| | |
|---|---|
| Base model | `__UPSTREAM__` (__PARAMS__B parameters) |
| Method | **full fine-tune**, all __TRAINABLE__ parameters, via Unsloth |
| Data | `Bangla-Instruct`, __TRAINN__ examples within __MAXSEQ__ tokens |
| Learning rate | __LR__, cosine schedule, 40 warmup steps |
| Batch | __BATCH__ × __ACCUM__ accumulation = __EFF__ effective |
| Steps | __STEPS__ |
| Optimizer | `adamw_8bit`, `max_grad_norm=1.0` |
| Precision | __PREC__ |
| Peak VRAM | __VRAM__ GB of 15.0 |
| Hardware | __GPU__ (free Kaggle tier) |
| Training time | ~__MINS__ minutes |

### Decisions that mattered

**Full fine-tuning, and why it fits.** Gemma-3-1B is roughly 2 GB in fp16; gradients add
2 GB more. Plain AdamW would need about 8 GB for its two fp32 moment buffers and would not
fit a T4. `adamw_8bit` brings the optimizer state to around 2 GB, and gradient checkpointing
absorbs the activations. At 3B this would not fit without quantising the base — at which
point it would be QLoRA, not full fine-tuning.

**Learning rate 1e-5, not 2e-4.** Every weight moves in a full fine-tune, so steps must be
small. LoRA tolerates 2e-4 because only a fraction of a percent of parameters carry the
update. Running a full fine-tune at LoRA's learning rate destroys the multilingual ability
the base model arrived with.

**Native Bangla data, not translated Alpaca.** Most Bangla fine-tunes use machine-translated
Alpaca derivatives, which carry English sentence structure and translation artefacts into
the model. `Bangla-Instruct` was generated natively in Bangla.

**Loss on the assistant turn only.** Prompt tokens are masked to `-100`, so the gradient is
not spent regenerating the question.

## Limitations

- **1B parameters.** Fluent Bangla does not mean accurate Bangla. This model will state
  false things confidently, particularly about dates, numbers and named people.
- **Multiple-choice benchmarks measure recognition, not generation.** A model can pick the
  right option and still write poor Bangla, or the reverse. The sample table exists because
  the benchmark table cannot answer that question.
- **__BENCH_N__ items per benchmark**, not the full test sets, to fit free GPU hours.
  Differences under about 0.03 are noise.
- **No safety, bias or faithfulness evaluation** was run, in Bangla or English.
- **English ability was not measured** after fine-tuning. Full fine-tuning on one language
  can degrade others, and that cost is unquantified here.
- **Religious, political and regional sensitivity** in Bangla content was not evaluated.
  Review output before publishing it.

## What a v2 would change

Each item addresses a specific number above, in order of expected effect:

1. **Continual pretraining on Bangla text before instruction tuning.** The largest lever, and
   the one TigerLLM used: 10M tokens of Bangla corpus first, instruction tuning second.
   Instruction tuning adapts form; it does not install knowledge.
2. **A 1024–2048 token window.** A 512-token limit kept __TRAINN__ of 40,000 sampled pairs
   out of 342,391 available.
3. **More steps on more data** — __STEPS__ steps saw roughly __SEEN__ examples.
4. **A benchmark that discriminates at this scale**, plus human reading of generations, which
   is what actually matters for a writing assistant.
5. **A 4B variant** once the recipe is settled, where factual capacity stops being the
   binding constraint.

## Reproducing

One Kaggle notebook on a __GPU__ produced the baseline, the training, every chart and this
card. Raw numbers are in [`results.json`](results.json).

## Citation

```bibtex
@inproceedings{raihan-zampieri-2025-tigerllm,
  title     = {{TigerLLM} - A Family of {B}angla Large Language Models},
  author    = {Raihan, Nishat and Zampieri, Marcos},
  booktitle = {Proceedings of ACL},
  year      = {2025}
}
@software{unsloth,
  title  = {Unsloth}, author = {Daniel Han and Michael Han and Unsloth team},
  year   = {2023}, url = {https://github.com/unslothai/unsloth}
}
```
"""

repl = {
 "__MODEL_NAME__": MODEL_NAME, "__UPSTREAM__": UPSTREAM, "__REPO_ID__": REPO_ID,
 "__GGUF_REPO__": GGUF_REPO, "__SYSTEM__": SYSTEM_BN, "__HEADLINE__": headline,
 "__BENCH_ROWS__": bench_rows, "__EX_ROWS__": ex_rows, "__GGUF_ROWS__": gguf_rows,
 "__PPL_ROW__": ppl_row, "__BENCH_N__": str(BENCH_N),
 "__AVG__": f"{TUNED_BENCH['average']:.3f}", "__BASE_AVG__": f"{BASE_BENCH['average']:.3f}",
 "__D_AVG__": f"{D_AVG:+.3f}",
 "__TPW__": f"{TOKENS_PER_WORD:.2f}", "__ETPW__": f"{en_tpw:.2f}",
 "__RATIO__": f"{TOKENS_PER_WORD/en_tpw:.1f}",
 "__PARAMS__": f"{N_PARAMS/1e9:.1f}", "__TRAINABLE__": f"{TRAINABLE:,}",
 "__TRAINN__": f"{len(ds):,}", "__MAXSEQ__": str(MAX_SEQ), "__LR__": f"{LR:g}",
 "__BATCH__": str(BATCH), "__ACCUM__": str(ACCUM), "__EFF__": str(BATCH*ACCUM),
 "__STEPS__": str(MAX_STEPS), "__PREC__": "bf16" if BF16 else "fp16",
 "__VRAM__": f"{VRAM_TRAIN:.1f}", "__GPU__": GPU, "__MINS__": f"{TRAIN_MIN:.0f}",
 "__PPLN__": str(len(PPL_TEXTS)),
 "__BASE_PPL__": f"{BASE_PPL:.2f}" if BASE_PPL else "n/a",
 "__TUNED_PPL__": f"{TUNED_PPL:.2f}" if TUNED_PPL else "n/a",
 "__SEEN__": f"{MAX_STEPS*BATCH*ACCUM:,}",
}
for k, v in repl.items():
    CARD = CARD.replace(k, v)
leftover = sorted(set(re.findall(r"__[A-Z0-9_]+__", CARD)))
assert not leftover, f"unresolved placeholders: {leftover}"
if not os.path.exists(f"{ASSETS}/banner.png"):
    CARD = re.sub(r'<img src="assets/banner\.png".*?>\n\n', "", CARD, flags=re.S)
    print("no banner.png in assets/ — <img> line removed")

open(f"{SAVE}/README.md","w",encoding="utf-8").write(CARD)
json.dump({"base_model":UPSTREAM,"sft_dataset":SFT_DATASET,"bench_items":BENCH_N,
           "base":BASE_BENCH,"tuned":TUNED_BENCH,"delta_average":D_AVG,
           "perplexity":{"base":BASE_PPL,"tuned":TUNED_PPL},
           "tokens_per_word":{"bangla":round(TOKENS_PER_WORD,3),"english":round(en_tpw,3)},
           "loss_curve":CURVE,"examples":EXAMPLES,
           "config":{"method":"full fine-tune via Unsloth","max_seq":MAX_SEQ,
                     "batch":BATCH,"accum":ACCUM,"steps":MAX_STEPS,"lr":LR,
                     "optim":"adamw_8bit","total_params":N_PARAMS,
                     "trainable_params":TRAINABLE,"peak_vram_gb":round(VRAM_TRAIN,2),
                     "train_minutes":round(TRAIN_MIN,1),"hardware":GPU,"seed":SEED,
                     "merged_gb":round(merged_gb,2),"gguf":GGUF_FILES}},
          open(f"{SAVE}/results.json","w"), indent=2, ensure_ascii=False)
print("card written\n"); print(CARD[:1200])

## 14. Push

In [ ]:
for f in ["README.md","results.json","prompt_format.json"]:
    shutil.copy(f"{SAVE}/{f}", f"{MERGED}/{f}")
shutil.copytree(ASSETS, f"{MERGED}/assets", dirs_exist_ok=True)

create_repo(REPO_ID, repo_type="model", private=False, exist_ok=True)
upload_folder(repo_id=REPO_ID, folder_path=MERGED,
              commit_message=f"Full fine-tune for Bangla; mean accuracy "
                             f"{BASE_BENCH['average']:.3f} -> {TUNED_BENCH['average']:.3f}")
print(f"https://huggingface.co/{REPO_ID}")

if GGUF_OK:
    gcard = f"""---
language: [bn, en]
license: gemma
base_model: {REPO_ID}
base_model_relation: quantized
pipeline_tag: text-generation
tags: [gguf, llama.cpp, bangla, bengali, gemma3]
---

# {MODEL_NAME}-GGUF

llama.cpp builds of [`{REPO_ID}`](https://huggingface.co/{REPO_ID}), a full fine-tune of
Gemma-3-1B for Bangla.

| File | Size |
|---|---|
{gguf_rows}

```bash
llama-cli -hf {GGUF_REPO}:Q4_K_M -p "রবীন্দ্রনাথ ঠাকুর কে ছিলেন?" -n 256
```

Benchmarks, baselines and limitations are on the
[model card](https://huggingface.co/{REPO_ID}). Quantization costs some accuracy relative
to the 16-bit weights; that cost was not separately measured.
"""
    open(f"{GGUF}/README.md","w",encoding="utf-8").write(gcard)
    create_repo(GGUF_REPO, repo_type="model", private=False, exist_ok=True)
    upload_folder(repo_id=GGUF_REPO, folder_path=GGUF,
                  commit_message="GGUF builds (q4_k_m, q8_0)")
    print(f"https://huggingface.co/{GGUF_REPO}")

## 15. ONNX and the Space — do these afterwards, not here

Exporting a 1B model to ONNX needs more RAM than it needs GPU, and doing it in the same
session as training risks losing everything above to an out-of-memory kill. Both steps are
quick and safe once the model is on the Hub:

**ONNX:** open [`onnx-community/convert-to-onnx`](https://huggingface.co/spaces/onnx-community/convert-to-onnx),
paste the model id and a write token. It runs the official `transformers.js` conversion
script on Hugging Face's own hardware and pushes `bangla-gemma-pro-ONNX` to your account,
with the exact file names and quantisation variants Transformers.js expects.

**Space:** create a **Static** Space (Gradio now needs a paid plan; static is free) and
upload a page that loads the ONNX build through Transformers.js. Since the model runs in
the visitor's browser there is no server, no cold start and no per-user cost.

Gemma-3 is a recent architecture, so the ONNX conversion may fail. If it does, the Space
can still be built on ZeroGPU instead, which runs the PyTorch weights directly.

In [ ]:
print(f"""
Model : https://huggingface.co/{REPO_ID}
GGUF  : {'https://huggingface.co/' + GGUF_REPO if GGUF_OK else 'not built this run'}

mean accuracy  {BASE_BENCH['average']:.3f} -> {TUNED_BENCH['average']:.3f}  ({D_AVG:+.3f})
perplexity     {BASE_PPL} -> {TUNED_PPL}
peak VRAM      {VRAM_TRAIN:.1f} GB
training       {TRAIN_MIN:.0f} min

Next: ONNX via onnx-community/convert-to-onnx, then a Static Space.
""")